# Эксперимент 2.6: Оптимизационная точность vs качество предсказания

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error

from oracles import REG_MODEL_NAMEL2Oracle, CLASS_MODEL_NAMEL2Oracle
from optimization import lbfgs

In [ ]:
# TODO: замени на свои данные. Пока синтетика.
rng = np.random.default_rng(11)
m, n = 5000, 250
A = rng.normal(size=(m, n))
w = rng.normal(size=n)
y_reg = A @ w + 0.2 * rng.normal(size=m)
y_clf = np.sign(A @ w + 0.2 * rng.normal(size=m))
y_clf[y_clf == 0] = 1

A_tr, A_te, y_reg_tr, y_reg_te = train_test_split(A, y_reg, test_size=0.2, random_state=42)
_, _, y_clf_tr, y_clf_te = train_test_split(A, y_clf, test_size=0.2, random_state=42)

A_tr_sp = sparse.csr_matrix(A_tr)
A_te_sp = sparse.csr_matrix(A_te)

In [ ]:
def make_oracles_for_train(y_train, is_clf):
    def matvec_Ax(x):
        return A_tr_sp.dot(x)
    def matvec_ATx(v):
        return A_tr_sp.T.dot(v)
    def matmat_ATsA(s):
        return A_tr_sp.T.dot(sparse.diags(s)).dot(A_tr_sp).toarray()
    regcoef = 1.0 / A_tr_sp.shape[0]
    if is_clf:
        return CLASS_MODEL_NAMEL2Oracle(matvec_Ax, matvec_ATx, matmat_ATsA, y_train, regcoef)
    return REG_MODEL_NAMEL2Oracle(matvec_Ax, matvec_ATx, matmat_ATsA, y_train, regcoef)

oracle_reg = make_oracles_for_train(y_reg_tr, is_clf=False)
oracle_clf = make_oracles_for_train(y_clf_tr, is_clf=True)
x0 = np.zeros(n)

In [ ]:
ls = {'method': 'Wolfe', 'c1': 1e-4, 'c2': 0.9, 'alpha_0': 1.0}
_, _, h_reg = lbfgs(oracle_reg, x0, tolerance=1e-8, max_iter=120, memory_size=10, line_search_options=ls, trace=True)
_, _, h_clf = lbfgs(oracle_clf, x0, tolerance=1e-8, max_iter=120, memory_size=10, line_search_options=ls, trace=True)

In [ ]:
def evaluate_path(path_x, is_clf):
    metrics = []
    for x in path_x:
        pred = A_te_sp.dot(x)
        if is_clf:
            y_hat = np.where(pred >= 0, 1, -1)
            metrics.append(accuracy_score(y_clf_te, y_hat))
        else:
            metrics.append(mean_squared_error(y_reg_te, pred))
    return np.array(metrics)

# Если x не сохранен в history (для больших n), аппроксимируем через перезапуск с trace x
_, _, h_reg_x = lbfgs(oracle_reg, x0, tolerance=1e-8, max_iter=40, memory_size=10, line_search_options=ls, trace=True)
_, _, h_clf_x = lbfgs(oracle_clf, x0, tolerance=1e-8, max_iter=40, memory_size=10, line_search_options=ls, trace=True)

if 'x' in h_reg_x and len(h_reg_x['x']) > 0:
    metric_reg = evaluate_path(h_reg_x['x'], is_clf=False)
else:
    metric_reg = np.array([])

if 'x' in h_clf_x and len(h_clf_x['x']) > 0:
    metric_clf = evaluate_path(h_clf_x['x'], is_clf=True)
else:
    metric_clf = np.array([])

In [ ]:
def plot_dual_axis(hist, metric, metric_name, title):
    g = np.array(hist['grad_norm'])
    it = np.arange(len(g))

    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(it, hist['func'], label='f_train')
    ax1.plot(it, g, label='||grad||')
    ax1.set_yscale('log')
    ax1.set_xlabel('iteration')
    ax1.set_ylabel('train objective / grad norm (log)')
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    if metric.size > 0:
        ax2.plot(np.arange(len(metric)), metric, color='tab:red', label=metric_name)
    ax2.set_ylabel(metric_name)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')
    plt.title(title)
    plt.tight_layout()

plot_dual_axis(h_reg, metric_reg, 'test MSE', 'Regression: optimization vs quality')
plot_dual_axis(h_clf, metric_clf, 'test Accuracy', 'Classification: optimization vs quality')